## Imports 

In [3]:
from Bio import SeqIO
from Bio.Seq import Seq
import json
import os

## Declares

In [4]:
# Empty

## Helper functions

In [19]:
# Helper functions
def gard_parser(combo, best_gard, codon_MSA_in):    
    data = [d.strip() for d in open(best_gard).readlines() if "CHARSET" in d]
    coords = []
    if len(data) > 1:
        for pos, i in enumerate(data):
            ## hyphy coordinates are 1 indexed and INCLUSIVE ##
            ## so to get to "true" python coords you need to -1 to start and -1 to stop ##
            ## then you need to +1 to python "stop" coord to get the actual length ## 
            #hyphy_start = i.split(' ')[-1].split(';')[0].split('-')[0]
            #hyphy_stop = i.split(" ")[-1].split(";")[0].split("-")[1]
            #print(f"HYPHY coords for partition {pos +1} --> start {hyphy_start} | stop {hyphy_stop}")
            ## in python (start-1) | (end-1) to get the true coords ##
            ## to get the correct python index[X:Y] (zero index, EXLUSIVE) need to do [start-1,stop] ##
            start = int(i.split(" ")[-1].split(";")[0].split("-")[0])-1
            stop = int(i.split(" ")[-1].split(";")[0].split("-")[1])-1
            coords.append((start,stop))        
        #end for
    ## print out fasta file
    else:
        print(data)
    #end if
    
    print("We are using 0-indexed coordinates:", coords, "\n")
    
    ## now use the coords to pull out seqs in the codon aware MSA ##
    index_data = {}
    for pos, c in enumerate(coords):
        temp_dict = {}
        ## want list of 3 ##
        n = 3
        recs = list(rec for rec in SeqIO.parse(codon_MSA_in, "fasta"))  
        
        ## actual codons ##
        index_map = [list(range(len(recs[0].seq)))[i*n:(i+1) *n] for i in range((len(list(recs[0].seq)) + n - 1) // n )]
        #print(index_map)
        old_start = c[0]
        old_stop = c[1]
        new_start = ''
        new_stop = ''
        
        ## getting new start ##
        for imap in index_map:
            if old_start in imap:
                for p, j in enumerate(imap):
                    if old_start == j:
                        if p !=0:
                            new_start = imap[2]+1
                        else:
                            new_start = imap[0]
                        #end if
                    #end if
                #end for
            elif old_stop in imap:
                new_stop = imap[0]
            else:
                continue 
            #end if
        #end for
        
        ## sanity check 
        print(f"# Sanity check: OLD {old_start}, {old_stop} | NEW {new_start}, {new_stop} | NEW div by 3 {(int(new_stop)-int(new_start))/3}")
        temp_dict[pos+1] = [i+1 for i in list(range(int(new_start),int(new_stop)))]
        index_data.update(temp_dict)
        
        if new_stop < old_stop:
            print("Add GAP codon")
        
        ### ~ CHANGE TO PLACE YOU WANT OUTPUT ~ ###
        #codon_out = os.path.join("..", "results", label, "%s.%s.codon.fas" % (combo, str(pos+1)))
        codon_out = os.path.join("..", "tests", label, "%s.%s.codon.fas" % (combo, str(pos+1)))
        
        # Save
        print("# Saving partition to:", codon_out, "\n")
        with open(codon_out, "w") as out_f:
            ## just grabbing one record right now ## 
            #for record in recs[0:1]:
            for record in recs:
                ## now grabbing actual partition ##
                partition = record[int(new_start):int(new_stop)] + Seq("---")
                ## double sanity check ##
                #print(len(partition),len(partition)/3)
                #print(partition.seq)
                out_f.write(">{}\n{}\n".format(partition.id, partition.seq))
            #end for
        #end with
    
    print(f"\nDone with {combo}\n")
#end method



# Main subroutine

In [20]:
"""
rule gard_parse:
    input:
        input_GARD = rules.recombination.output.output,
        input_MSA = rules.strike_ambigs.output.out_strike_ambigs
    params:
        genelabel = Label
    output:
        output = os.path.join(OUTDIR, Label + ".1.codon.fas")
    notebook:
        "notebooks/GARD_Parser.ipynb"
#end rule

"""



# Input from the snakemake pipeline.
#label =  snakemake.params.genelabel
label = "mammalian_REV3L"

# Grab the best-gard and input-msa, we need these.
# We are going from the best-gard charsets,
# And restoring the full-fasta file with breakpoints
# As were determined from the downsampled fasta.
best_gard = os.path.join("..", "results", label, label + "_codons.SA.fasta.cluster.fasta.best-gard")
input_msa = os.path.join("..", "results", label, label + "_codons.SA.fasta")

# Print to user
print("# Processing best-gard:", best_gard)
print("# Input Alignment:", input_msa)

# check input alignment length
first_record = next(SeqIO.parse(input_msa, "fasta"))
print("# Input MSA length:", len(first_record.seq))

# Do work here.
gard_parser(label, best_gard, input_msa)


# Processing best-gard: ../results/mammalian_REV3L/mammalian_REV3L_codons.SA.fasta.cluster.fasta.best-gard
# Input Alignment: ../results/mammalian_REV3L/mammalian_REV3L_codons.SA.fasta
# Input MSA length: 10884
We are using 0-indexed coordinates: [(0, 215), (216, 594), (595, 674), (675, 3682), (3683, 6932), (6933, 7559), (7560, 7712), (7713, 7873), (7874, 8126), (8127, 8253), (8254, 8577), (8578, 9060), (9061, 9623), (9624, 10883)] 

# Sanity check: OLD 0, 215 | NEW 0, 213 | NEW div by 3 71.0
Add GAP codon
# Saving partition to: ../tests/mammalian_REV3L/mammalian_REV3L.1.codon.fas 

# Sanity check: OLD 216, 594 | NEW 216, 594 | NEW div by 3 126.0
# Saving partition to: ../tests/mammalian_REV3L/mammalian_REV3L.2.codon.fas 

# Sanity check: OLD 595, 674 | NEW 597, 672 | NEW div by 3 25.0
Add GAP codon
# Saving partition to: ../tests/mammalian_REV3L/mammalian_REV3L.3.codon.fas 

# Sanity check: OLD 675, 3682 | NEW 675, 3681 | NEW div by 3 1002.0
Add GAP codon
# Saving partition to: ../tes

## End of file